In [15]:
import pandas as pd
import pymongo

df = pd.read_csv('data.csv')
df.head()

,id,Gender,Age,Driving_License,Region_Code,Previously_Insured,Vehicle_Age,Vehicle_Damage,Annual_Premium,Policy_Sales_Channel,Vintage,Response
0,1,Male,44,1,28.0,0,> 2 Years,Yes,40454.0,26.0,217,1
1,2,Male,76,1,3.0,0,1-2 Year,No,33536.0,26.0,183,0
2,3,Male,47,1,28.0,0,> 2 Years,Yes,38294.0,26.0,27,1
3,4,Male,21,1,11.0,1,< 1 Year,No,28619.0,152.0,203,0
4,5,Female,29,1,41.0,1,< 1 Year,No,27496.0,152.0,39,0


In [19]:
# df should be converted into dict before we push it to mongodb

data = df.to_dict(orient='records')
# data

In [ ]:
import os
from pathlib import Path

import certifi
from dotenv import load_dotenv

# Load .env from project root (notebook runs from Notebook/)
load_dotenv(Path("..") / ".env")

CONNECTION_URL = os.getenv("MONGODB_URL")
if not CONNECTION_URL:
    raise ValueError("MONGODB_URL not set. Add it to .env in the project root.")

DB_NAME = "Proj1"
COLLECTION_NAME = "Proj1-Data"

In [21]:
# Network diagnostic — run this if connection fails
import socket
import ssl
import urllib.request

import dns.resolver

public_ip = urllib.request.urlopen("https://api.ipify.org", timeout=10).read().decode()
host = "ac-bmjzxd2-shard-00-00.huhict3.mongodb.net"

tcp_ok = False
tls_ok = False
try:
    socket.create_connection((host, 27017), timeout=10).close()
    tcp_ok = True
except OSError as exc:
    tcp_error = str(exc)

if tcp_ok:
    try:
        ctx = ssl.SSLContext(ssl.PROTOCOL_TLS_CLIENT)
        ctx.check_hostname = False
        ctx.verify_mode = ssl.CERT_NONE
        sock = socket.create_connection((host, 27017), timeout=10)
        tls = ctx.wrap_socket(sock, server_hostname=host)
        tls_ok = True
        tls.close()
    except ssl.SSLError as exc:
        tls_error = str(exc)

print(f"Your public IP: {public_ip}")
print(f"TCP to Atlas ({host}:27017): {'OK' if tcp_ok else 'FAILED'}")
print(f"TLS handshake: {'OK' if tls_ok else 'FAILED'}")

if tcp_ok and not tls_ok:
    print(
        "\nAtlas is reachable but rejecting TLS. This almost always means "
        "your IP is NOT whitelisted in MongoDB Atlas.\n"
        f"Add this IP in Atlas -> Network Access -> Add IP Address: {public_ip}\n"
        "Or use 0.0.0.0/0 (allow from anywhere) for development, then wait ~1 min and re-run."
    )

Your public IP: 203.99.50.227
TCP to Atlas (ac-bmjzxd2-shard-00-00.huhict3.mongodb.net:27017): OK
TLS handshake: OK


In [22]:
import pymongo
from pymongo.errors import ServerSelectionTimeoutError

client = pymongo.MongoClient(
    CONNECTION_URL,
    tlsCAFile=certifi.where(),
    serverSelectionTimeoutMS=30000,
)
data_base = client[DB_NAME]
collection = data_base[COLLECTION_NAME]

try:
    client.admin.command("ping")
except ServerSelectionTimeoutError as exc:
    if "SSL handshake failed" in str(exc):
        raise RuntimeError(
            "MongoDB Atlas rejected the connection during TLS handshake. "
            "Run the diagnostic cell above, then whitelist your public IP in "
            "Atlas -> Security -> Network Access -> Add IP Address."
        ) from exc
    raise

print("Connected to MongoDB Atlas")

Connected to MongoDB Atlas


In [23]:
# Uploading data to MongoDB
rec = collection.insert_many(data)

In [7]:
# Load back data from mongodb

df = pd.DataFrame(list(collection.find()))
df.head(2)

,_id,id,Gender,Age,Driving_License,Region_Code,Previously_Insured,Vehicle_Age,Vehicle_Damage,Annual_Premium,Policy_Sales_Channel,Vintage,Response
0,6730464b17bb55425068a27e,1,Male,44,1,28.0,0,> 2 Years,Yes,40454.0,26.0,217,1
1,6730464b17bb55425068a27f,2,Male,76,1,3.0,0,1-2 Year,No,33536.0,26.0,183,0


In [ ]:
## If you still get ServerSelectionTimeoutError / SSL handshake failed

1. **Whitelist your IP in MongoDB Atlas** (most common cause):
   - Atlas → Network Access → Add IP Address
   - Add your current public IP, or `0.0.0.0/0` for development only
2. Confirm the cluster is **running** (not paused).
3. Ensure `MONGO_URI` in `.env` uses the correct username/password.